<a href="https://colab.research.google.com/github/KelvinM9187/wav2vec_unsupervised/blob/main/wav2vec_prosit3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# access to google drive
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/wav2vec_unsupervised

Mounted at /content/drive
/content/drive/MyDrive/wav2vec_unsupervised


In [ ]:
# fix setup_function
%%bash
cat << 'EOF' > setup_functions.sh
#!/bin/bash
set -e
set -o pipefail

INSTALL_ROOT="/content/drive/MyDrive/wav2vec_unsupervised"
FAIRSEQ_ROOT="$INSTALL_ROOT/fairseq_"
VENV_PATH="$INSTALL_ROOT/venv"
log() { echo "[$(date +'%Y-%m-%d %H:%M:%S')] $1"; }

basic_dependencies() {
    log "Installing dependencies..."
    sudo apt-get update && sudo apt-get install -y python3-venv libatlas3-base cmake dos2unix git
}

create_dirs() {
    log "Creating project directories..."
    mkdir -p "$INSTALL_ROOT/data/logs/zulu_project"
}

setup_venv() {
    log "Setting up virtual environment..."
    python3 -m venv --without-pip "$VENV_PATH"
    source "$VENV_PATH/bin/activate"
    curl -sS https://bootstrap.pypa.io/get-pip.py | python3
    deactivate
}

install_pytorch_and_other_packages() {
    log "Installing PyTorch..."
    source "$VENV_PATH/bin/activate"
    pip install torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 --index-url https://download.pytorch.org/whl/cu121
}

install_fairseq() {
    log "Installing Fairseq..."
    source "$VENV_PATH/bin/activate"
    pip install "pip<24.1" --quiet
    pip install "omegaconf==2.0.6"
    if [ ! -d "$FAIRSEQ_ROOT" ]; then
        git clone https://github.com/Ashesi-Org/fairseq_.git "$FAIRSEQ_ROOT"
    fi
    cd "$FAIRSEQ_ROOT" && pip install --editable ./ --no-deps
    pip install cffi cython regex sacrebleu tqdm bitarray scikit-learn packaging
}

install_flashlight() {
    log "Installing Flashlight..."
    source "$VENV_PATH/bin/activate"
    pip install flashlight-text
}

install_kenlm() { log "Placeholder for KenLM"; }
install_rVADfast() { log "Placeholder for rVADfast"; }
download_pretrained_model() { log "Downloading wav2vec_vox_new.pt"; }
download_languageIdentification_model() { log "Downloading LID model"; }
EOF

bash -n setup_functions.sh && echo "Syntax OK"

Syntax OK


In [ ]:
%%bash
# Fix paths in utils.sh
sed -i "s|DIR_PATH=.*|DIR_PATH=\"/content/drive/MyDrive/wav2vec_unsupervised\"|" utils.sh
sed -i 's/LANG="en"/LANG="zu"/' utils.sh
sed -i 's/DATASET_NAME="librispeech"/DATASET_NAME="zulu_project"/' utils.sh
sed -i 's|VENV_PATH="/content/venv"|VENV_PATH="/content/drive/MyDrive/wav2vec_unsupervised/venv"|' utils.sh

echo "=== Verifying fixes ==="
grep "DIR_PATH\|LANG\|DATASET_NAME\|VENV_PATH" utils.sh | head -10

chmod +x run_setup.sh
./run_setup.sh

=== Verifying fixes ===
DIR_PATH="/content/drive/MyDrive/wav2vec_unsupervised"
DATA_ROOT="$DIR_PATH/data" # a folder that stores all the data generated from pipeline
FAIRSEQ_ROOT="$DIR_PATH/fairseq_" # the root directory of the fairseq repository
KENLM_ROOT="$DIR_PATH/kenlm/build/bin"  # Path to KenLM installation
VENV_PATH="/content/drive/MyDrive/wav2vec_unsupervised/venv"
RVAD_ROOT="$DIR_PATH/rVADfast/src/rVADfast" # the root directory of the rVADfast repository
SPEECHPROCS="$DIR_PATH/rVADfast/src/rVADfast/speechproc/speechproc.py"
OPENFST_PATH="$DIR_PATH/fairseq/examples/speech_recognition/kaldi/kaldi_initializer.py"
LANG="zu"
FASTTEXT_LIB_MODEL="$DIR_PATH/lid_model/lid.176.bin"  # the path to the language identification model
[2026-04-17 11:34:11] Installing dependencies...
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
dos2unix: converting file eval_functions.sh to Unix format...
dos2unix: converting file gans_functions.sh to Unix format...
dos2unix: converting file run_eval.sh to Unix format...
dos2unix: converting file run_gans.sh to Unix format...
dos2unix: converting file run_setup.sh to Unix format...
dos2unix: converting file run_wav2vec.sh to Unix format...
dos2unix: converting file setup_functions.sh to Unix format...
dos2unix: converting file utils.sh to Unix format...
dos2unix: converting file wav2vec_functions.sh to Unix format...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
    PyYAML (>=5.1.*)
            ~~~~~~^
    PyYAML (>=5.1.*)
            ~~~~~~^
    PyY

In [ ]:
!pip install datasets soundfile librosa --quiet

In [ ]:
import os
import soundfile as sf
import numpy as np
from datasets import load_dataset

OUTPUT_DIR = "/content/drive/MyDrive/zulu_audio"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading 200 samples from NCHLT isiZulu dataset...")
dataset = load_dataset("danielshaps/nchlt_speech_zul", split="train[:200]", trust_remote_code=True)
print(f"Loaded {len(dataset)} samples")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'danielshaps/nchlt_speech_zul' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'danielshaps/nchlt_speech_zul' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading 200 samples from NCHLT isiZulu dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00006.parquet:   0%|          | 0.00/445M [00:00<?, ?B/s]

...

data/train-00005-of-00006.parquet:   0%|          | 0.00/605M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/224M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/41871 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2802 [00:00<?, ? examples/s]

Loaded 200 samples


In [ ]:
import librosa

TARGET_SR = 16000
saved = 0

for i, sample in enumerate(dataset):
    try:
        audio_array = np.array(sample['audio']['array'])
        original_sr = sample['audio']['sampling_rate']

        if original_sr != TARGET_SR:
            audio_array = librosa.resample(audio_array, orig_sr=original_sr, target_sr=TARGET_SR)

        if audio_array.ndim > 1:
            audio_array = audio_array.mean(axis=0)

        sf.write(os.path.join(OUTPUT_DIR, f"zulu_{i:05d}.wav"), audio_array, TARGET_SR, subtype='PCM_16')
        saved += 1

        if i % 50 == 0:
            print(f"  [{i}/200] saved {saved} files...")

    except Exception as e:
        print(f"Skipped {i}: {e}")

print(f"\nDone! Saved: {saved} wav files to {OUTPUT_DIR}")

  [0/200] saved 1 files...
  [50/200] saved 51 files...
  [100/200] saved 101 files...
  [150/200] saved 151 files...

Done! Saved: 200 wav files to /content/drive/MyDrive/zulu_audio


In [ ]:
%%bash
cd /content/drive/MyDrive

# Skip if already downloaded
if [ -f "zulu_text.txt" ] && [ $(wc -l < zulu_text.txt) -gt 100 ]; then
    echo "zulu_text.txt already exists, skipping download"
    wc -l zulu_text.txt
else
    echo "Downloading Zulu Wikipedia..."
    wget -q https://dumps.wikimedia.org/zuwiki/latest/zuwiki-latest-pages-articles.xml.bz2 \
         -O zuwiki.xml.bz2 --show-progress
    pip install wikiextractor -q
    python -m wikiextractor.WikiExtractor zuwiki.xml.bz2 \
        -o zuwiki_extracted --no-templates --quiet
    find zuwiki_extracted -name "wiki_*" -exec cat {} \; \
        | grep -v "^<" | grep -v "^$" \
        | sed 's/\. /\n/g' > zulu_text.txt
    echo "Done! Lines:"
    wc -l zulu_text.txt
fi

zulu_text.txt already exists, skipping download
200 zulu_text.txt


In [ ]:
%%bash
AUDIO_SRC="/content/drive/MyDrive/zulu_audio"
TRAIN_DIR="/content/drive/MyDrive/zulu_audio_split/train"
VAL_DIR="/content/drive/MyDrive/zulu_audio_split/val"
TEST_DIR="/content/drive/MyDrive/zulu_audio_split/test"

mkdir -p $TRAIN_DIR $VAL_DIR $TEST_DIR

FILES=($AUDIO_SRC/*.wav)

for i in "${!FILES[@]}"; do
    f="${FILES[$i]}"
    if [ $i -lt 160 ]; then
        cp "$f" "$TRAIN_DIR/"
    elif [ $i -lt 180 ]; then
        cp "$f" "$VAL_DIR/"
    else
        cp "$f" "$TEST_DIR/"
    fi
done

echo "Train: $(ls $TRAIN_DIR | wc -l) files"
echo "Val:   $(ls $VAL_DIR | wc -l) files"
echo "Test:  $(ls $TEST_DIR | wc -l) files"

Train: 160 files
Val:   20 files
Test:  20 files


In [ ]:
%%bash
echo "=== WAV FILES ==="
echo "Train: $(ls /content/drive/MyDrive/zulu_audio_split/train/*.wav 2>/dev/null | wc -l)"
echo "Val:   $(ls /content/drive/MyDrive/zulu_audio_split/val/*.wav 2>/dev/null | wc -l)"
echo "Test:  $(ls /content/drive/MyDrive/zulu_audio_split/test/*.wav 2>/dev/null | wc -l)"

echo ""
echo "=== TEXT FILE ==="
wc -l /content/drive/MyDrive/zulu_text.txt

echo ""
echo "=== VENV ==="
ls /content/drive/MyDrive/wav2vec_unsupervised/venv/bin/python3

echo ""
echo "=== ALL GOOD - READY TO RUN PIPELINE ==="

=== WAV FILES ===
Train: 160
Val:   20
Test:  20

=== TEXT FILE ===
200 /content/drive/MyDrive/zulu_text.txt

=== VENV ===
/content/drive/MyDrive/wav2vec_unsupervised/venv/bin/python3

=== ALL GOOD - READY TO RUN PIPELINE ===


In [ ]:
%%bash
# Find the GAN related files in the repo
find /content/drive/MyDrive/wav2vec_unsupervised/fairseq_ -name "*.py" | xargs grep -l "discriminator\|generator\|GAN\|gan" -i 2>/dev/null

/content/drive/MyDrive/wav2vec_unsupervised/fairseq_/examples/MMPT/mmpt/models/mmfusionnlg.py
/content/drive/MyDrive/wav2vec_unsupervised/fairseq_/examples/adaptive_span/truncated_bptt_lm_task.py
/content/drive/MyDrive/wav2vec_unsupervised/fairseq_/examples/attention_head_selection/src/speech_to_text_head_selection.py
/content/drive/MyDrive/wav2vec_unsupervised/fairseq_/examples/criss/save_encoder.py
/content/drive/MyDrive/wav2vec_unsupervised/fairseq_/examples/data2vec/models/modalities/base.py
/content/drive/MyDrive/wav2vec_unsupervised/fairseq_/examples/discriminative_reranking_nmt/drnmt_rerank.py
/content/drive/MyDrive/wav2vec_unsupervised/fairseq_/examples/discriminative_reranking_nmt/tasks/discriminative_reranking_task.py
/content/drive/MyDrive/wav2vec_unsupervised/fairseq_/examples/emotion_conversion/preprocess/build_hifigan_manifest.py
/content/drive/MyDrive/wav2vec_unsupervised/fairseq_/examples/emotion_conversion/preprocess/create_core_manifest.py
/content/drive/MyDrive/wav2v

In [ ]:
%%bash
echo "=== CORE WAV2VEC UNSUPERVISED FOLDER ==="
ls /content/drive/MyDrive/wav2vec_unsupervised/fairseq_/examples/wav2vec/unsupervised/

echo ""
echo "=== MODELS FOLDER (GAN code lives here) ==="
ls /content/drive/MyDrive/wav2vec_unsupervised/fairseq_/examples/wav2vec/unsupervised/models/

echo ""
echo "=== TASKS FOLDER ==="
ls /content/drive/MyDrive/wav2vec_unsupervised/fairseq_/examples/wav2vec/unsupervised/tasks/

=== CORE WAV2VEC UNSUPERVISED FOLDER ===
config
data
__init__.py
kaldi_self_train
models
README.md
scripts
tasks
w2vu_generate.py

=== MODELS FOLDER (GAN code lives here) ===
__init__.py
wav2vec_u.py
wav2vec_u.py.bak

=== TASKS FOLDER ===
__init__.py
unpaired_audio_text.py


In [ ]:
%%bash
cat /content/drive/MyDrive/wav2vec_unsupervised/fairseq_/examples/wav2vec/unsupervised/models/wav2vec_u.py

# [PASTE YOUR FULL REFACTORED CODE HERE]


In [ ]:
%%bash
cat << 'ENDOFFILE' > /content/drive/MyDrive/wav2vec_unsupervised/fairseq_/examples/wav2vec/unsupervised/models/wav2vec_u.py
# Copyright (c) Facebook, Inc. and its affiliates.
#
# This source code is licensed under the MIT license found in the
# LICENSE file in the root directory of this source tree.

# Refactored by: Kelvin Mhodi
# Refactoring: Separation of Concerns - Generator, RealData, Discriminator

import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, field
from fairseq import utils
from fairseq.models import BaseFairseqModel, register_model
from fairseq.dataclass import FairseqDataclass
from omegaconf import MISSING


# ============================================================
#  CONFIGURATION
# ============================================================

@dataclass
class Wav2VecUConfig(FairseqDataclass):
    """Configuration for the Wav2Vec-U GAN model."""

    # Generator settings
    generator_dim: int = field(
        default=256,
        metadata={"help": "Hidden dimension size inside the Generator"}
    )
    generator_layers: int = field(
        default=3,
        metadata={"help": "Number of convolutional layers in the Generator"}
    )
    generator_kernel: int = field(
        default=5,
        metadata={"help": "Kernel size for Generator convolutions"}
    )
    generator_dropout: float = field(
        default=0.0,
        metadata={"help": "Dropout applied inside the Generator"}
    )
    no_xavier_init: bool = field(
        default=False,
        metadata={"help": "If True, skip Xavier weight initialization"}
    )

    # Discriminator settings
    discriminator_dim: int = field(
        default=256,
        metadata={"help": "Hidden dimension inside the Discriminator"}
    )
    discriminator_layers: int = field(
        default=4,
        metadata={"help": "Number of layers in the Discriminator"}
    )
    discriminator_kernel: int = field(
        default=3,
        metadata={"help": "Kernel size for Discriminator convolutions"}
    )
    discriminator_dropout: float = field(
        default=0.0,
        metadata={"help": "Dropout applied inside the Discriminator"}
    )
    discriminator_spectral_norm: bool = field(
        default=False,
        metadata={"help": "Apply spectral normalization to Discriminator weights"}
    )
    discriminator_weight_norm: bool = field(
        default=False,
        metadata={"help": "Apply weight normalization to Discriminator weights"}
    )

    # Training loss weights
    gradient_penalty: float = field(
        default=0.0,
        metadata={"help": "Weight for gradient penalty regularization term"}
    )
    smoothness_weight: float = field(
        default=0.0,
        metadata={"help": "Weight for smoothness regularization on Generator output"}
    )
    code_penalty: float = field(
        default=0.0,
        metadata={"help": "Penalty to encourage diverse phoneme code usage"}
    )

    # Segmentation / output
    segmentation_type: str = field(
        default="random",
        metadata={"help": "Type of segmentation: 'random' or 'uniform'"}
    )


# ============================================================
#  CLASS 1: Generator
#  Responsibility: Take audio features (fake/unlabelled speech)
#  and generate a sequence of pseudo-phonemes that looks like
#  real text to fool the Discriminator.
# ============================================================

class Generator(nn.Module):
    """
    The Generator receives audio feature vectors extracted by wav2vec
    and maps them to a probability distribution over phonemes.
    It is the 'fake' side of the GAN — trying to produce outputs
    that the Discriminator cannot distinguish from real text phonemes.
    """

    def __init__(self, input_dim: int, output_dim: int, cfg: Wav2VecUConfig):
        super().__init__()
        self.cfg = cfg

        # Stack of 1D convolutional layers to process the audio feature sequence
        layers = []
        in_dim = input_dim
        for i in range(cfg.generator_layers):
            out_dim = cfg.generator_dim
            layers.append(
                nn.Conv1d(
                    in_channels=in_dim,
                    out_channels=out_dim,
                    kernel_size=cfg.generator_kernel,
                    padding=cfg.generator_kernel // 2,  # keep sequence length same
                )
            )
            layers.append(nn.GELU())  # smooth non-linearity used in transformers
            if cfg.generator_dropout > 0:
                layers.append(nn.Dropout(p=cfg.generator_dropout))
            in_dim = out_dim

        self.conv_layers = nn.Sequential(*layers)

        # Final projection: map hidden dim → number of phoneme classes
        self.output_projection = nn.Linear(cfg.generator_dim, output_dim)

        # Optionally initialize weights with Xavier uniform
        if not cfg.no_xavier_init:
            self._init_weights()

    def _init_weights(self):
        """Xavier uniform initialization for stable GAN training."""
        for module in self.modules():
            if isinstance(module, (nn.Conv1d, nn.Linear)):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)

    def forward(self, audio_features: torch.Tensor) -> torch.Tensor:
        """
        Args:
            audio_features: Tensor of shape (Batch, Time, Feature_dim)
                            — raw wav2vec features from unlabelled speech
        Returns:
            logits: Tensor of shape (Batch, Time, num_phonemes)
                    — unnormalized scores over phoneme vocabulary
        """
        # Conv1d expects (Batch, Channels, Time) so we transpose
        x = audio_features.transpose(1, 2)   # (B, F, T)
        x = self.conv_layers(x)               # (B, hidden_dim, T)
        x = x.transpose(1, 2)                # (B, T, hidden_dim)
        logits = self.output_projection(x)    # (B, T, num_phonemes)
        return logits


# ============================================================
#  CLASS 2: RealData
#  Responsibility: Load and provide REAL text phoneme sequences
#  to the Discriminator so it can learn what genuine Zulu
#  phoneme patterns look like.
# ============================================================

class RealData(nn.Module):
    """
    RealData encapsulates the real text side of the GAN.
    It takes a vocabulary of phonemes (from zulu_text.txt after
    prepare_text processing) and embeds them into dense vectors
    that the Discriminator can process alongside Generator outputs.

    This class cleanly separates the 'real sample provider'
    from the Generator and Discriminator, following the
    principle of Separation of Concerns.
    """

    def __init__(self, vocab_size: int, embed_dim: int):
        super().__init__()
        # Learned embedding table: maps phoneme token IDs to dense vectors
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0,  # index 0 is reserved for padding
        )
        self.embed_dim = embed_dim

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        """
        Args:
            token_ids: LongTensor of shape (Batch, SeqLen)
                       — integer IDs of real phoneme tokens from text corpus
        Returns:
            embeddings: Tensor of shape (Batch, SeqLen, embed_dim)
                        — dense real phoneme representations
        """
        return self.embedding(token_ids)   # (B, T, embed_dim)

    def get_embeddings(self) -> nn.Embedding:
        """Return the raw embedding table (used by Discriminator for scoring)."""
        return self.embedding


# ============================================================
#  CLASS 3: Discriminator
#  Responsibility: Given a sequence of phoneme vectors — either
#  from the Generator (fake) or from RealData (real) — output
#  a score indicating how 'real' the sequence looks.
# ============================================================

class Discriminator(nn.Module):
    """
    The Discriminator is a binary classifier that receives phoneme
    sequences and must decide: is this real Zulu text (from the
    corpus) or fake output (from the Generator)?

    As training proceeds, the Generator gets better at fooling
    the Discriminator, which forces both to improve — this is
    the core GAN dynamic driving unsupervised speech recognition.
    """

    def __init__(self, input_dim: int, cfg: Wav2VecUConfig):
        super().__init__()
        self.cfg = cfg

        def make_conv(in_c, out_c, k):
            """Helper to build a conv layer with optional normalization."""
            conv = nn.Conv1d(in_c, out_c, kernel_size=k, padding=k // 2)
            if cfg.discriminator_spectral_norm:
                conv = nn.utils.spectral_norm(conv)
            elif cfg.discriminator_weight_norm:
                conv = nn.utils.weight_norm(conv)
            return conv

        layers = []
        in_dim = input_dim
        for _ in range(cfg.discriminator_layers):
            out_dim = cfg.discriminator_dim
            layers.append(make_conv(in_dim, out_dim, cfg.discriminator_kernel))
            layers.append(nn.LeakyReLU(0.2))   # standard GAN activation
            if cfg.discriminator_dropout > 0:
                layers.append(nn.Dropout(p=cfg.discriminator_dropout))
            in_dim = out_dim

        self.conv_layers = nn.Sequential(*layers)

        # Final layer outputs a single real/fake score per time step
        self.output_layer = nn.Linear(cfg.discriminator_dim, 1)

    def forward(self, phoneme_vectors: torch.Tensor) -> torch.Tensor:
        """
        Args:
            phoneme_vectors: Tensor of shape (Batch, Time, embed_dim)
                             — either from Generator or RealData
        Returns:
            scores: Tensor of shape (Batch, Time, 1)
                    — higher score = Discriminator thinks it is more 'real'
        """
        x = phoneme_vectors.transpose(1, 2)   # (B, embed_dim, T)
        x = self.conv_layers(x)               # (B, disc_dim, T)
        x = x.transpose(1, 2)                # (B, T, disc_dim)
        scores = self.output_layer(x)         # (B, T, 1)
        return scores


# ============================================================
#  MAIN MODEL: Wav2VecU
#  Wires together Generator, RealData, and Discriminator
#  following fairseq model conventions.
# ============================================================

@register_model("wav2vec_u", dataclass=Wav2VecUConfig)
class Wav2VecU(BaseFairseqModel):
    """
    Wav2Vec-U: Unsupervised Speech Recognition using GANs.

    Architecture (Separation of Concerns):
      - Generator:     fake phoneme sequences from audio features
      - RealData:      real phoneme sequences from text corpus
      - Discriminator: tells real from fake

    Reference: Baevski et al., 2021 - Unsupervised Speech Recognition
    """

    def __init__(self, cfg: Wav2VecUConfig, generator, real_data, discriminator):
        super().__init__()
        self.cfg = cfg

        # Each concern is a clearly named, independently testable component
        self.generator = generator
        self.real_data = real_data
        self.discriminator = discriminator

    @classmethod
    def build_model(cls, cfg: Wav2VecUConfig, task):
        """
        Factory method called by fairseq to construct the model.
        Reads vocabulary and feature sizes from the task object.
        """
        # Sizes come from the task (set up by unpaired_audio_text.py)
        feature_dim = task.cfg.input_dim if hasattr(task.cfg, 'input_dim') else 512
        vocab_size = len(task.target_dictionary)
        embed_dim = cfg.generator_dim

        # Instantiate each concern separately — clean and testable
        generator = Generator(
            input_dim=feature_dim,
            output_dim=vocab_size,
            cfg=cfg,
        )
        real_data = RealData(
            vocab_size=vocab_size,
            embed_dim=embed_dim,
        )
        discriminator = Discriminator(
            input_dim=embed_dim,
            cfg=cfg,
        )

        return cls(cfg, generator, real_data, discriminator)

    def forward(self, audio_features, real_tokens=None, training_phase="generator"):
        """
        One forward pass through the GAN.

        Args:
            audio_features:  wav2vec features from unlabelled speech (B, T, F)
            real_tokens:     phoneme token IDs from text corpus (B, T) — optional
            training_phase:  'generator' or 'discriminator'

        Returns:
            dict with generator logits and/or discriminator scores
        """
        result = {}

        # --- Generator forward pass (always runs) ---
        gen_logits = self.generator(audio_features)        # (B, T, vocab)
        result["generator_logits"] = gen_logits

        # Convert logits to soft embeddings for Discriminator input
        gen_probs = F.softmax(gen_logits, dim=-1)          # (B, T, vocab)
        # Weighted sum of real_data embeddings using generator probs
        embed_weight = self.real_data.get_embeddings().weight  # (vocab, embed_dim)
        fake_vectors = gen_probs @ embed_weight            # (B, T, embed_dim)

        # --- Discriminator scores fake samples ---
        fake_scores = self.discriminator(fake_vectors)     # (B, T, 1)
        result["fake_scores"] = fake_scores

        # --- Discriminator scores real samples (when training discriminato

bash: line 359: warning: here-document at line 1 delimited by end-of-file (wanted `ENDOFFILE')


In [ ]:
%%bash
head -20 /content/drive/MyDrive/wav2vec_unsupervised/fairseq_/examples/wav2vec/unsupervised/models/wav2vec_u.py
echo "..."
tail -10 /content/drive/MyDrive/wav2vec_unsupervised/fairseq_/examples/wav2vec/unsupervised/models/wav2vec_u.py

# Copyright (c) Facebook, Inc. and its affiliates.
#
# This source code is licensed under the MIT license found in the
# LICENSE file in the root directory of this source tree.

# Refactored by: [YOUR NAME]
# Refactoring: Separation of Concerns - Generator, RealData, Discriminator

import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, field
from fairseq import utils
from fairseq.models import BaseFairseqModel, register_model
from fairseq.dataclass import FairseqDataclass
from omegaconf import MISSING


# ============================================================
#  CONFIGURATION
...
        gen_probs = F.softmax(gen_logits, dim=-1)          # (B, T, vocab)
        # Weighted sum of real_data embeddings using generator probs
        embed_weight = self.real_data.get_embeddings().weight  # (vocab, embed_dim)
        fake_vectors = gen_probs @ embed_weight            # (B, T, embed_dim)

        # --- Discriminator scores fake samples ---


In [ ]:
%%bash
# The path where the original wav2vec_u.py lives
MODEL_PATH="/content/drive/MyDrive/wav2vec_unsupervised/fairseq_/examples/wav2vec/unsupervised/models/wav2vec_u.py"

# Create a backup of the original just in case
cp "$MODEL_PATH" "${MODEL_PATH}.bak"

# Overwrite it with your refactored version (the code you just showed me)
# NOTE: Replace 'final_model_prosit3.py' with the name of the file you uploaded if it's different
cat << 'EOF' > "$MODEL_PATH"
# [PASTE YOUR FULL REFACTORED CODE HERE]
EOF

echo "Model refactored and saved to $MODEL_PATH"

Model refactored and saved to /content/drive/MyDrive/wav2vec_unsupervised/fairseq_/examples/wav2vec/unsupervised/models/wav2vec_u.py


In [ ]:
%%bash
ls /content/drive/MyDrive/wav2vec_unsupervised/

cuda_installation.png
cuda_installation.txt
data
eval_functions.sh
fairseq_
gans_functions.sh
README.md
run_eval.sh
run_gans.sh
run_setup.sh
run_wav2vec.sh
setup_functions.sh
utils.sh
vads.py
venv
wav2vec_functions.sh


In [ ]:
# Change these two lines in utils.sh
LANG="zu"
DATASET_NAME="zulu_project"


In [ ]:
#!/bin/bash
set -e
set -o pipefail

source utils.sh
source gans_functions.sh

create_dirs
activate_venv
setup_path

log "Starting GAN training for $DATASET_NAME"

train_gans

log "GAN training pipeline completed successfully!"


In [ ]:
#!/bin/bash
set -e
set -o pipefail

source utils.sh
source eval_functions.sh

create_dirs
activate_venv

# Auto-find the best checkpoint from the multirun directory
log "Searching for best checkpoint in $RESULTS_DIR..."

CHECKPOINT_PATH=$(find "$RESULTS_DIR" -name "checkpoint_best.pt" | sort | tail -1)

if [ -z "$CHECKPOINT_PATH" ]; then
    log "ERROR: No checkpoint_best.pt found in $RESULTS_DIR"
    log "Make sure run_gans.sh completed successfully first"
    exit 1
fi

log "Found checkpoint: $CHECKPOINT_PATH"

# Override MODEL_PATH with the found checkpoint
MODEL_PATH="$CHECKPOINT_PATH"
export MODEL_PATH

log "Starting evaluation for $DATASET_NAME"

transcription_gans_viterbi

log "Evaluation completed! Transcriptions saved to $GANS_OUTPUT_PHONES"


In [ ]:
%%bash
sed -i 's/LANG="en"/LANG="zu"/' /content/drive/MyDrive/wav2vec_unsupervised/utils.sh
sed -i 's/DATASET_NAME="librispeech"/DATASET_NAME="zulu_project"/' /content/drive/MyDrive/wav2vec_unsupervised/utils.sh

# Verify
grep "LANG\|DATASET_NAME" /content/drive/MyDrive/wav2vec_unsupervised/utils.sh


In [ ]:
%%bash
BASE="/content/drive/MyDrive/wav2vec_unsupervised"

# Overwrite the three files
cp /path/to/downloaded/utils.sh "$BASE/utils.sh"
cp /path/to/downloaded/run_gans.sh "$BASE/run_gans.sh"
cp /path/to/downloaded/run_eval.sh "$BASE/run_eval.sh"

echo "Done. Verifying..."
grep "LANG\|DATASET_NAME" "$BASE/utils.sh"


In [ ]:
%%bash
cd /content/drive/MyDrive/wav2vec_unsupervised

bash run_wav2vec.sh \
  /content/drive/MyDrive/zulu_audio_split/train \
  /content/drive/MyDrive/zulu_audio_split/val \
  /content/drive/MyDrive/zulu_audio_split/test \
  /content/drive/MyDrive/zulu_text.txt


In [ ]:
%%bash
cd /content/drive/MyDrive/wav2vec_unsupervised

bash run_gans.sh


In [ ]:
%%bash
cd /content/drive/MyDrive/wav2vec_unsupervised

bash run_eval.sh
